In [2]:
"""
=========================================================
EXTRACCIÓN DE FEATURES — FRAIWAN (DIAGNÓSTICO VACÍO)
=========================================================
Procesa TODOS los .wav de la carpeta, sin intentar deducir
el diagnóstico. La columna 'diagnostico' se deja VACÍA para
que la rellenes tú manualmente (útil para los audios
multietiqueta como "Asthma y Lung Fibrosis").

Para ayudarte a rellenar, además se añaden dos columnas
informativas extraídas del nombre del archivo:
  - patient_id_raw : lo que va antes del primer "_"
  - etiqueta_cruda : lo que va entre "_" y la primera ","

ASÍ, en el CSV verás algo como:
  archivo                                  | diagnostico | patient_id_raw | etiqueta_cruda
  BP1_Asthma and Lung Fibrosis,I E,...wav  |  (vacío)    | BP1            | Asthma and Lung Fibrosis

Tú rellenas la columna 'diagnostico' y, cuando termines,
puedes borrar las dos columnas auxiliares antes de pasar
al script de modelos.

Mismo pipeline que ICBHI: 4 kHz, ventanas de 5 s, umbral 1e-5,
41 features.
=========================================================
"""

import os
import numpy as np
import librosa
import pandas as pd
from scipy.stats import skew, kurtosis

# =========================================================
# RUTAS
# =========================================================

audio_dir   = r"C:\Users\josem\Desktop\tfg\AudioFiles"
output_path = r"C:\Users\josem\Desktop\tfg\features_fraiwan_sin_diagnostico.csv"

# =========================================================
# PARÁMETROS (idénticos a ICBHI)
# =========================================================

TARGET_SR        = 4000
WINDOW_SECONDS   = 5
window_size      = TARGET_SR * WINDOW_SECONDS
N_MFCC           = 13
SILENCE_THRESHOLD = 1e-5

# =========================================================
# PARSER DEL NOMBRE (solo informativo)
# =========================================================

def parse_filename(filename):
    """Devuelve (patient_id_raw, etiqueta_cruda) a partir del nombre."""
    nombre = os.path.splitext(filename)[0]
    if "_" not in nombre:
        return nombre, ""
    patient_id, resto = nombre.split("_", 1)
    if "," in resto:
        etiqueta = resto.split(",", 1)[0]
    else:
        etiqueta = resto
    return patient_id.strip(), etiqueta.strip()

# =========================================================
# FEATURES DE ENTROPÍA
# =========================================================

def shannon_entropy(signal):
    n      = len(signal)
    sigma  = np.std(signal)
    bw     = 3.5 * sigma / (n ** (1/3))
    bins   = max(int((np.max(signal) - np.min(signal)) / (bw + 1e-12)), 10)
    hist, _= np.histogram(signal, bins=bins, density=True)
    p      = (hist + 1e-12) / np.sum(hist + 1e-12)
    return -np.sum(p * np.log2(p))

def log_energy_entropy(signal):
    hist, _= np.histogram(signal, bins=64, density=True)
    p      = (hist + 1e-12) / np.sum(hist + 1e-12)
    return -np.sum((np.log(p)) ** 2)

def spectral_entropy(signal):
    S   = np.abs(librosa.stft(signal, n_fft=2048)) ** 2
    psd = np.sum(S, axis=1)
    psd = psd / (np.sum(psd) + 1e-12)
    return -np.sum(psd * np.log2(psd + 1e-12))

# =========================================================
# EXTRACCIÓN DE FEATURES
# =========================================================

def extraer_features(window, sr):
    feats = {}

    feats['shannon_entropy']    = shannon_entropy(window)
    feats['log_energy_entropy'] = log_energy_entropy(window)
    feats['spectral_entropy']   = spectral_entropy(window)

    mfccs = librosa.feature.mfcc(y=window, sr=sr, n_mfcc=N_MFCC)
    for i in range(N_MFCC):
        feats[f'mfcc_{i+1}_mean'] = np.mean(mfccs[i])
        feats[f'mfcc_{i+1}_std']  = np.std(mfccs[i])

    zcr = librosa.feature.zero_crossing_rate(window)
    feats['zcr_mean'] = np.mean(zcr)
    feats['zcr_std']  = np.std(zcr)

    rms = librosa.feature.rms(y=window)
    feats['rms_mean'] = np.mean(rms)
    feats['rms_std']  = np.std(rms)

    sc = librosa.feature.spectral_centroid(y=window, sr=sr)
    feats['spectral_centroid_mean'] = np.mean(sc)
    feats['spectral_centroid_std']  = np.std(sc)

    sb = librosa.feature.spectral_bandwidth(y=window, sr=sr)
    feats['spectral_bandwidth_mean'] = np.mean(sb)
    feats['spectral_bandwidth_std']  = np.std(sb)

    sr_feat = librosa.feature.spectral_rolloff(y=window, sr=sr)
    feats['spectral_rolloff_mean'] = np.mean(sr_feat)
    feats['spectral_rolloff_std']  = np.std(sr_feat)

    feats['skewness'] = skew(window)
    feats['kurtosis'] = kurtosis(window)

    return feats

# =========================================================
# PROCESAMIENTO — TODOS LOS ARCHIVOS
# =========================================================

resultados            = []
errores               = []
descartados_silencio  = []
etiquetas_encontradas = {}   # para mostrarte un resumen al final

print(f"Carpeta de audios: {audio_dir}")
print(f"Umbral de silencio: {SILENCE_THRESHOLD}")
print("Procesando TODOS los archivos (diagnóstico se deja vacío).")
print("-" * 60)

for file in sorted(os.listdir(audio_dir)):

    if not file.lower().endswith(".wav"):
        continue

    file_path = os.path.join(audio_dir, file)
    patient_id_raw, etiqueta_cruda = parse_filename(file)

    # Registrar la etiqueta cruda para el resumen final
    etiquetas_encontradas[etiqueta_cruda] = etiquetas_encontradas.get(etiqueta_cruda, 0) + 1

    try:
        y, sr = librosa.load(file_path, sr=TARGET_SR)

        window_count = 0

        for start in range(0, len(y), window_size):
            end    = start + window_size
            window = y[start:end]

            if len(window) < window_size:
                continue
            if np.mean(window ** 2) < SILENCE_THRESHOLD:
                continue

            feats = extraer_features(window, sr)

            feats['archivo']        = file
            feats['diagnostico']    = ""              # <-- VACÍO, lo rellenas tú
            feats['ventana']        = window_count
            feats['patient_id_raw'] = patient_id_raw
            feats['etiqueta_cruda'] = etiqueta_cruda

            resultados.append(feats)
            window_count += 1

        if window_count == 0:
            descartados_silencio.append(file)
        print(f"  {file}  ({window_count} ventanas)  [etiqueta cruda: {etiqueta_cruda}]")

    except Exception as e:
        errores.append((file, str(e)))
        print(f"  Error en {file}: {e}")

# =========================================================
# RESUMEN
# =========================================================

print()
print("=" * 60)
print("RESUMEN")
print("=" * 60)

print("\nEtiquetas crudas encontradas en los nombres de archivo:")
for etiq, n in sorted(etiquetas_encontradas.items()):
    print(f"   '{etiq}'  ->  {n} archivos")

if descartados_silencio:
    print(f"\nArchivos con todas las ventanas silenciosas: {len(descartados_silencio)}")
    for f in descartados_silencio:
        print(f"   {f}")

if errores:
    print(f"\nArchivos con error: {len(errores)}")
    for f, e in errores:
        print(f"   {f}: {e}")

# =========================================================
# GUARDAR CSV
# =========================================================

if len(resultados) == 0:
    print("\nERROR: no se ha generado ninguna fila. Revisa la ruta de audios.")
else:
    df = pd.DataFrame(resultados)

    # Orden de columnas: meta + auxiliares primero, features después
    meta_cols = ['archivo', 'diagnostico', 'ventana',
                 'patient_id_raw', 'etiqueta_cruda']
    feature_cols = [c for c in df.columns if c not in meta_cols]
    df = df[meta_cols + feature_cols]

    df.to_csv(output_path, index=False, sep=';', decimal=',')

    print(f"\n{'=' * 60}")
    print(f"CSV guardado en: {output_path}")
    print(f"Features por ventana:  {len(feature_cols)}")
    print(f"Total ventanas:        {len(df)}")
    print(f"Archivos únicos:       {df['archivo'].nunique()}")

Carpeta de audios: C:\Users\josem\Desktop\tfg\AudioFiles
Umbral de silencio: 1e-05
Procesando TODOS los archivos (diagnóstico se deja vacío).
------------------------------------------------------------
  BP100_N,N,P R M,70,F.wav  (2 ventanas)  [etiqueta cruda: N]
  BP101_Asthma,E W,P L M,12,F.wav  (2 ventanas)  [etiqueta cruda: Asthma]
  BP102_N,N,P L L,41,M.wav  (3 ventanas)  [etiqueta cruda: N]
  BP103_N,N,P R U,81,F.wav  (2 ventanas)  [etiqueta cruda: N]
  BP104_Asthma,E W,P L U,45,F.wav  (2 ventanas)  [etiqueta cruda: Asthma]
  BP105_Lung Fibrosis,Crep,A U R,44,M.wav  (5 ventanas)  [etiqueta cruda: Lung Fibrosis]
  BP106_Asthma,E W,P L U,45,F.wav  (3 ventanas)  [etiqueta cruda: Asthma]
  BP107_Asthma,E W,P L U,59,F.wav  (2 ventanas)  [etiqueta cruda: Asthma]
  BP108_COPD,E W,P R L ,63,M.wav  (4 ventanas)  [etiqueta cruda: COPD]
  BP109_N,N,P L M,26,M.wav  (2 ventanas)  [etiqueta cruda: N]
  BP10_Asthma,E W,P R U,59,M.wav  (1 ventanas)  [etiqueta cruda: Asthma]
  BP110_COPD,E W,P L